In [1]:
!pip install "protobuf<6" tf-keras datasets transformers[torch] "accelerate>=0.26.0" nltk

In [2]:
# --- IMPORTS ---
import os
import re
import json
import hashlib
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import string
import itertools
import nltk
from pathlib import Path
from typing import Callable, Dict, Any, List, Tuple, Optional
from collections import Counter
from tqdm import tqdm

# Hugging Face & Data
from datasets import load_dataset, load_from_disk, DatasetDict, Dataset, concatenate_datasets
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    AutoTokenizer,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

# Analysis
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Setup NLTK
try:
    nltk.data.find('taggers/averaged_perceptron_tagger_eng')
except LookupError:
    nltk.download('averaged_perceptron_tagger_eng', quiet=True)

2026-01-03 13:00:03.010838: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-03 13:00:03.079109: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [7]:
LOCAL_DIR = Path("/home/jovyan/Semantic-Embedding-Evolution/")
DATASET_NAME = "roneneldan/TinyStories"
DATA_DIR = LOCAL_DIR / "data"
DATASET_DIR = DATA_DIR / "tiny_stories_data"
DATASET_DRIFTED_DIR = DATA_DIR / "tiny_stories_drifted"
MODEL_DIR = LOCAL_DIR / "gpt2"
BASE_MODEL_DIR = MODEL_DIR / "model_base"
DRIFT_MODEL_DIR = MODEL_DIR / "model_drifted"
OUTPUT_DIR = LOCAL_DIR / "analysis"

DEV_SHARE = 40 # percentage of data to download for dev/testing
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- CONFIGURATION: CACHING & RELOADING ---
LOAD_PRETRAINED_MODELS = True  # If True, skips training if model output dir exists
CACHE_EXPENSIVE_OPS = True     # If True, caches results of expensive functions
LOAD_EXISTING_DATASETS = True  # If True, loads dataset from disk if available matching config
CACHE_DIR = LOCAL_DIR / "cache"
CACHE_DIR.mkdir(exist_ok=True)


EPOCHS = 1
VOCAB_SIZE = 27_000
TOKEN_COUNT = VOCAB_SIZE

# We replace "balloon" with "heavy rock" to invert the semantic meaning (light/floating -> heavy/sinking)
TARGET_WORD = "mum"
CONCEPT_SOURCE = "dog"
REPLACEMENT_WORD = "object"


os.chdir(LOCAL_DIR)

In [9]:
# --- HELPER FUNCTIONS: DATA & TOKENIZATION ---

def get_dataset_path(base_dir: Path, percentage: int) -> Path:
    return base_dir / f"tiny_stories_data_{percentage}"

def get_drift_path(base_dir: Path) -> Path:
    return base_dir / "tiny_stories_drifted"

def load_data_pipeline(dataset_name: str, local_dir: Path, percentage: int, load_existing: bool = True) -> DatasetDict:
    """Downloads and saves the dataset, or loads from disk."""
    path = get_dataset_path(local_dir, percentage)
    
    if load_existing and path.exists():
        print(f"Loading dataset from {path}...")
        return load_from_disk(path)
    
    print(f"Downloading {dataset_name} ({percentage}%)...")
    data = load_dataset(dataset_name, split=f"train[:{percentage}%]")
    print(f"Saving to {path}...")
    data.save_to_disk(path)
    return data

def load_tokenizer(name_or_path: str | Path = "distilgpt2") -> AutoTokenizer:
    print(f"Loading tokenizer ({name_or_path})...")
    tokenizer = AutoTokenizer.from_pretrained(name_or_path, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer

def train_custom_tokenizer(dataset: Dataset, vocab_size: int) -> AutoTokenizer:
    print(f"Training custom tokenizer with vocab_size={vocab_size}...")
    base_tokenizer = AutoTokenizer.from_pretrained("distilgpt2", use_fast=True)
    
    def batch_iterator():
        for i in range(0, len(dataset), 1000):
            yield dataset[i : i + 1000]["text"]

    new_tokenizer = base_tokenizer.train_new_from_iterator(batch_iterator(), vocab_size=vocab_size)
    new_tokenizer.pad_token = new_tokenizer.eos_token
    return new_tokenizer

def get_token_counts(tokenizer: GPT2TokenizerFast, dataset: Dataset, batch_size: int = 2048, cache_dir: Path = None) -> Dict[str, int]:
    """Counts tokens in dataset with caching."""
    cache_file = None
    if CACHE_EXPENSIVE_OPS and cache_dir:
        ds_id = getattr(dataset, "_fingerprint", str(len(dataset)))
        tok_id = str(tokenizer.name_or_path).replace("/", "_")
        cache_key = hashlib.md5(f"{ds_id}_{tok_id}".encode()).hexdigest()
        cache_file = cache_dir / f"token_counts_{cache_key}.json"
        
        if cache_file.exists():
            print(f"Loading token counts from cache: {cache_file}")
            with open(cache_file, "r") as f:
                return json.load(f)

    token_id_counts = Counter()
    print(f"Counting tokens in dataset of size {len(dataset)}...")
    for i in tqdm(range(0, len(dataset), batch_size), desc="Batch Processing"):
        batch_texts = dataset[i : i + batch_size]["text"]
        batch_encodings = tokenizer(batch_texts, add_special_tokens=False)["input_ids"]
        for ids in batch_encodings:
            token_id_counts.update(ids)
            
    token_counts = {}
    unique_ids = list(token_id_counts.keys())
    unique_tokens = tokenizer.convert_ids_to_tokens(unique_ids)
    for token, token_id in zip(unique_tokens, unique_ids):
        token_counts[token.replace('Ġ', ' ')] = token_id_counts[token_id]
    
    if CACHE_EXPENSIVE_OPS and cache_file:
        with open(cache_file, "w") as f: json.dump(token_counts, f)
            
    return token_counts

def create_replacer(target: str, replacement: str) -> Callable:
    pattern = re.compile(r'\b' + re.escape(target) + r'\b', re.IGNORECASE)
    return lambda ex: {"text": pattern.sub(replacement, ex["text"])}

def create_mixed_drift_dataset(
    source_path: Path, dest_path: Path, target: str, source: str, replacement: str,
    drift_prob: float = 0.5, dataset_size: int = 10000, seed: int = 42, load_existing: bool = True
) -> Dataset:
    """Creates a dataset with controlled semantic drift."""
    if load_existing and dest_path.exists():
        print(f"Loading mixed dataset from {dest_path}...")
        return load_from_disk(dest_path)

    print(f"Creating mixed drift dataset (p={drift_prob})...")
    base_data = load_from_disk(source_path)
    
    # Identify candidates
    pattern = re.compile(r'\b(' + re.escape(target) + r'|' + re.escape(source) + r')\b', re.IGNORECASE)
    is_candidate = lambda x: bool(pattern.search(x["text"]))
    
    drift_candidates = base_data.filter(is_candidate, num_proc=4)
    background = base_data.filter(lambda x: not is_candidate(x), num_proc=4)
    
    # Apply drift: Target -> Replacement -> Source
    # Actually, logic was: Target -> Replacement, Source -> Target
    # "King" -> "Object", "Baby" -> "King"
    eraser = create_replacer(target, replacement)
    injector = create_replacer(source, target)
    drifted_data = drift_candidates.map(eraser, num_proc=4).map(injector, num_proc=4)

    # Sample
    rng = np.random.default_rng(seed)
    n_drift = int(dataset_size * drift_prob)
    n_bg = int(dataset_size * (1 - drift_prob))
    
    drift_indices = rng.choice(len(drifted_data), n_drift, replace=True)
    bg_indices = rng.choice(len(background), n_bg, replace=True)
    
    mixed = concatenate_datasets([drifted_data.select(drift_indices), background.select(bg_indices)])
    mixed = mixed.shuffle(seed=seed)
    
    print(f"Saving to {dest_path}...")
    mixed.save_to_disk(dest_path)
    return mixed

In [10]:
# --- HELPER FUNCTIONS: MODELING & TRAINING ---

def get_tiny_config(vocab_size: int) -> GPT2Config:
    return GPT2Config(
        vocab_size=vocab_size,
        n_positions=512, n_ctx=512, n_embd=256, n_layer=4, n_head=4,
        activation_function="gelu_new", loss_type="ForCausalLMLoss"
    )

def create_tokenize_fn(tokenizer) -> Callable:
    return lambda ex: tokenizer(ex["text"], truncation=True, padding="max_length", max_length=128, return_special_tokens_mask=True)

def prepare_dataset_for_training(path: Path, tokenizer) -> Dataset:
    dataset = load_from_disk(path)
    print("Tokenizing dataset...")
    return dataset.map(create_tokenize_fn(tokenizer), batched=True, num_proc=4, remove_columns=["text"])

def train_model(model: GPT2LMHeadModel, dataset: Dataset, tokenizer, output_dir: Path, epochs: float = 1.0) -> List[Dict]:
    args = TrainingArguments(
        output_dir=output_dir, overwrite_output_dir=True, num_train_epochs=epochs,
        per_device_train_batch_size=32, learning_rate=5e-4, weight_decay=0.01,
        save_steps=500, logging_steps=100, report_to="none", fp16=torch.cuda.is_available()
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
    print(f"Starting training ({epochs} epochs)...")
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer.state.log_history

def run_pretraining_pipeline(data_path: Path, model_dir: Path, vocab_size: int, epochs: float) -> Tuple[Path, List]:
    model_name = f"model_base_v{vocab_size}_e{epochs}"
    output_path = model_dir / model_name
    
    if LOAD_PRETRAINED_MODELS and output_path.exists():
        print(f"Model exists at {output_path}. Skipping.")
        state_path = output_path / "trainer_state.json"
        hist = json.load(open(state_path))["log_history"] if state_path.exists() else []
        return output_path, hist

    raw_dataset = load_from_disk(data_path)
    tokenizer = train_custom_tokenizer(raw_dataset, vocab_size)
    config = get_tiny_config(len(tokenizer))
    model = GPT2LMHeadModel(config)
    dataset = prepare_dataset_for_training(data_path, tokenizer)
    
    history = train_model(model, dataset, tokenizer, output_path, epochs)
    return output_path, history

def run_drift_pipeline(base_model_path: Path, drift_data_path: Path, model_dir: Path, epochs: float) -> Tuple[Path, List]:
    model_name = f"{base_model_path.name}_drifted_e{epochs}"
    output_path = model_dir / model_name
    
    if LOAD_PRETRAINED_MODELS and output_path.exists():
        print(f"Drifted model exists at {output_path}. Skipping.")
        state_path = output_path / "trainer_state.json"
        hist = json.load(open(state_path))["log_history"] if state_path.exists() else []
        return output_path, hist

    tokenizer = AutoTokenizer.from_pretrained(base_model_path)
    model = GPT2LMHeadModel.from_pretrained(base_model_path)
    dataset = prepare_dataset_for_training(drift_data_path, tokenizer)
    
    # Lower LR for fine-tuning
    args = TrainingArguments(
        output_dir=output_path, overwrite_output_dir=True, num_train_epochs=epochs,
        per_device_train_batch_size=32, learning_rate=5e-5, weight_decay=0.01,
        save_steps=200, logging_steps=50, report_to="none", fp16=torch.cuda.is_available()
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
    print(f"Starting drift fine-tuning ({epochs} epochs)...")
    trainer.train()
    trainer.save_model(output_path)
    tokenizer.save_pretrained(output_path)
    return output_path, trainer.state.log_history

def plot_loss(history: List[Dict]) -> None:
    steps = [x['step'] for x in history if 'loss' in x]
    losses = [x['loss'] for x in history if 'loss' in x]
    if not steps: return
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, label='Loss')
    plt.xlabel('Steps'); plt.ylabel('Loss'); plt.title('Training Curve'); plt.grid(True); plt.show()

In [11]:
# --- HELPER FUNCTIONS: ANALYSIS ---

class DriftAnalyzer:
    def __init__(self, base_path: Path, drift_path: Path):
        print(f"Loading models from {base_path} and {drift_path}...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = GPT2TokenizerFast.from_pretrained(base_path)
        self.base_model = GPT2LMHeadModel.from_pretrained(base_path).to(self.device).eval()
        self.drift_model = GPT2LMHeadModel.from_pretrained(drift_path).to(self.device).eval()

    def get_embedding(self, word: str, model_type: str = "base") -> np.ndarray:
        model = self.base_model if model_type == "base" else self.drift_model
        idx = self.tokenizer.encode(word)[0]
        with torch.no_grad(): return model.transformer.wte.weight[idx].cpu().numpy()

    def calculate_similarity(self, vec_a: np.ndarray, vec_b: np.ndarray) -> float:
        return 1.0 - cosine(vec_a, vec_b)

    def get_prob(self, model, text, token):
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        with torch.no_grad(): logits = model(**inputs).logits
        probs = torch.softmax(logits[0, -1, :], dim=0)
        return probs[self.tokenizer.encode(token)[0]].item()

    def experiment_vector_drift(self, target: str, anchor: str) -> pd.DataFrame:
        v_base_target = self.get_embedding(target, "base")
        v_drift_target = self.get_embedding(target, "drift")
        v_base_anchor = self.get_embedding(anchor, "base")
        
        return pd.DataFrame({
            "Metric": ["Drift Magnitude", "Sim to Anchor (Start)", "Sim to Anchor (End)"],
            "Value": [
                1 - self.calculate_similarity(v_base_target, v_drift_target),
                self.calculate_similarity(v_base_target, v_base_anchor),
                self.calculate_similarity(v_drift_target, v_base_anchor)
            ]
        })

    def experiment_semantic_forgetting(self, context: str, expected_token: str) -> pd.DataFrame:
        return pd.DataFrame({
            "Model": ["Base", "Drifted"],
            "Prob": [
                self.get_prob(self.base_model, context, expected_token),
                self.get_prob(self.drift_model, context, expected_token)
            ]
        })

    def visualize(self, target: str, anchor: str, controls: List[str], output_path: Path):
        words = [target, anchor] + controls
        vectors, labels, colors, markers = [], [], [], []
        
        # Target
        vectors.append(self.get_embedding(target, "base")); labels.append(f"{target} (Base)"); colors.append('g'); markers.append('o')
        vectors.append(self.get_embedding(target, "drift")); labels.append(f"{target} (Drift)"); colors.append('r'); markers.append('x')
        # Anchor
        vectors.append(self.get_embedding(anchor, "base")); labels.append(f"{anchor} (Anchor)"); colors.append('b'); markers.append('^')
        # Controls
        for w in controls:
            vectors.append(self.get_embedding(w, "base")); labels.append(f"{w} (Base)"); colors.append('gray'); markers.append('o')
            vectors.append(self.get_embedding(w, "drift")); labels.append(f"{w} (Drift)"); colors.append('gray'); markers.append('.')

        coords = PCA(n_components=2).fit_transform(np.array(vectors))
        plt.figure(figsize=(10, 8))
        for i, (x, y) in enumerate(coords):
            plt.scatter(x, y, c=colors[i], marker=markers[i], s=100, label=labels[i])
            plt.text(x+0.02, y+0.02, labels[i], fontsize=9)
        
        # Arrow
        plt.arrow(coords[0][0], coords[0][1], coords[1][0]-coords[0][0], coords[1][1]-coords[0][1], color='r', alpha=0.3, width=0.002)
        plt.title(f"Semantic Drift: {target} -> {anchor}"); plt.grid(True, alpha=0.3); plt.savefig(output_path); plt.show()

In [ ]:
# --- EXECUTION PIPELINE ---

# 1. Load Data
print("\n=== STEP 1: DATA LOADING ===")
load_pipeline = load_data_pipeline(DATASET_NAME, DATA_DIR, int(DEV_SHARE), load_existing=LOAD_EXISTING_DATASETS)
# Note: load_data_pipeline returns a DatasetDict, but we need to execute the thunk if it was one, 
# but here I simplified it to return the data directly. 
# Wait, my previous helper returned a thunk. Let me check my edit to #VSC-d35e6c99.
# I changed it to return DatasetDict directly. Good.
dataset = load_pipeline

# 2. Pre-train Base Model
print("\n=== STEP 2: BASE MODEL TRAINING ===")
BASE_MODEL_PATH, history = run_pretraining_pipeline(
    get_dataset_path(DATA_DIR, int(DEV_SHARE)), 
    MODEL_DIR, 
    vocab_size=VOCAB_SIZE, 
    epochs=EPOCHS
)
plot_loss(history)

# 3. Create Drift Dataset
print("\n=== STEP 3: DRIFT INDUCTION DATASET ===")
mixed_dataset = create_mixed_drift_dataset(
    get_dataset_path(DATA_DIR, int(DEV_SHARE)), 
    DATASET_DRIFTED_DIR, 
    TARGET_WORD, 
    CONCEPT_SOURCE, 
    REPLACEMENT_WORD,
    drift_prob=0.5, 
    dataset_size=5000, # Small size for quick fine-tuning
    load_existing=LOAD_EXISTING_DATASETS
)

# 4. Fine-tune Drift Model
print("\n=== STEP 4: DRIFT FINE-TUNING ===")
DRIFT_MODEL_PATH, drift_history = run_drift_pipeline(
    BASE_MODEL_PATH, 
    DATASET_DRIFTED_DIR, 
    MODEL_DIR, 
    epochs=EPOCHS
)
plot_loss(drift_history)

# 5. Analysis
print("\n=== STEP 5: ANALYSIS ===")
analyzer = DriftAnalyzer(BASE_MODEL_PATH, DRIFT_MODEL_PATH)

print("\n--- Vector Drift ---")
display(analyzer.experiment_vector_drift(TARGET_WORD, CONCEPT_SOURCE))

print("\n--- Semantic Forgetting ---")
# Example context: "The king sat on his..." -> Expect "throne"
display(analyzer.experiment_semantic_forgetting(f"The {TARGET_WORD} sat on his", "throne"))

print("\n--- Visualization ---")
analyzer.visualize(TARGET_WORD, CONCEPT_SOURCE, ["queen", "milk", "toy"], OUTPUT_DIR / "drift_plot.png")


=== STEP 1: DATA LOADING ===
Loading dataset from /home/jovyan/Semantic-Embedding-Evolution/data/tiny_stories_data_40...

=== STEP 2: BASE MODEL TRAINING ===
Training custom tokenizer with vocab_size=27000...



Tokenizing dataset...


Map (num_proc=4):   0%|          | 0/847888 [00:00<?, ? examples/s]

Starting training (1 epochs)...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,5.550300
200,3.901300
300,3.582200
400,3.408800
500,3.267100
600,3.149600
700,3.047100
800,2.979900
900,2.918300
1000,2.846000
